# Sepedi Fine-Tuning — TRAINING notebook (train + validation only, test set is never evaluated here)

## Kaggle Setup

In [ ]:
import os
import sys

REPO_NAME = "A_Blockchain_Enabled_Framework_for_Misinformation_Monitoring"
REPO_URL = "https://github.com/SnowWoofer/A_Blockchain_Enabled_Framework_for_Misinformation_Monitoring.git"

if os.path.exists("/kaggle"):
    BASE_DIR = f"/kaggle/working/{REPO_NAME}"
else:
    BASE_DIR = "."

if not os.path.exists(BASE_DIR):
    !git clone {REPO_URL} {BASE_DIR}

os.chdir(BASE_DIR)
sys.path.append(os.path.join(BASE_DIR, "src"))

In [ ]:
# Only install what's actually missing -- do NOT run pip install -r requirements.txt on Kaggle,
# it will pull in torch/numpy/scipy versions that conflict with Kaggle's pre-matched base image.
import importlib
for pkg in ["torch", "transformers", "datasets", "evaluate", "sklearn"]:
    try:
        importlib.import_module(pkg)
        print(f"{pkg}: OK")
    except ImportError:
        print(f"{pkg}: MISSING")
        !pip install -q {pkg}

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print(torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Check Settings > Accelerator > GPU in the Kaggle sidebar.")

## Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import evaluate
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

seed = 100
np.random.seed(seed)
torch.manual_seed(seed)

## Load Sepedi Data — EDIT THE PATH BELOW to match your uploaded Kaggle Dataset
Splits into train / val / **held-out test (never evaluated in this notebook)**.

In [ ]:
# EDIT: replace with your actual uploaded Sepedi dataset path + filename
SEPEDI_CSV_PATH = "/kaggle/input/YOUR-SEPEDI-DATASET-NAME/twitter_data_nso_gpt_anon.csv"

df = pd.read_csv(SEPEDI_CSV_PATH, encoding="utf-8")
df = df.dropna(subset=["translated", "label"]).reset_index(drop=True)
df = df.rename(columns={"translated": "text"})

# If your Sepedi CSV doesn't already have a train/test 'set' column, create one here:
if "set" not in df.columns:
    train_idx, test_idx = train_test_split(
        df.index, test_size=0.1, stratify=df["label"], random_state=seed
    )
    df["set"] = "train"
    df.loc[test_idx, "set"] = "test"

train_full_df = df[df["set"] == "train"][["text", "label"]].reset_index(drop=True)
test_df = df[df["set"] == "test"][["text", "label"]].reset_index(drop=True)

# Carve validation out of train -- test_df stays untouched from here on
train_df, val_df = train_test_split(
    train_full_df, test_size=0.1, stratify=train_full_df["label"], random_state=seed
)

print(f"train: {len(train_df)}, val: {len(val_df)}, test (held out, untouched): {len(test_df)}")
print(train_df["label"].value_counts())

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

# Save the held-out test set to disk so the SEPARATE final-evaluation notebook
# uses this exact same split, not a re-derived one.
os.makedirs("/kaggle/working/afroxlmr_sepedi_final", exist_ok=True)
test_df.to_csv("/kaggle/working/afroxlmr_sepedi_final/held_out_test_set.csv", index=False)
print("Saved held-out test set for the separate evaluation notebook.")

## Load Model — starting from your English-fine-tuned checkpoint (Kaggle Model input)
EDIT THE PATH BELOW to match your uploaded English model.

In [ ]:
# EDIT: replace with your actual uploaded English-baseline Kaggle Model path
model_path = "/kaggle/input/YOUR-ENGLISH-MODEL-NAME/afroxlmr_eng_baseline_final"

id2label = {0: "not misinformation", 1: "misinformation"}
label2id = {"not misinformation": 0, "misinformation": 1}

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
# Freeze all base model params except the pooling layer (near the classification head)
for name, param in model.base_model.named_parameters():
    param.requires_grad = "pooler" in name

## Tokenization

In [ ]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Metrics

In [ ]:
accuracy_metric = evaluate.load("accuracy")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    recall = recall_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "recall": recall["recall"], "f1": f1["f1"]}

## Sanity-Check Run (tiny subset, 1 epoch) — run this first before the full run

In [ ]:
sanity_train = tokenized_train.select(range(min(100, len(tokenized_train))))
sanity_val = tokenized_val.select(range(min(50, len(tokenized_val))))

sanity_args = TrainingArguments(
    output_dir="/kaggle/working/sanity_check_sepedi",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_steps=5,
    report_to="none",
)

sanity_trainer = Trainer(
    model=model,
    args=sanity_args,
    train_dataset=sanity_train,
    eval_dataset=sanity_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

sanity_trainer.train()

## Full Training Run — only run after the sanity check above succeeds. Evaluates against VALIDATION only.

In [ ]:
output_dir = "/kaggle/working/afroxlmr_sepedi"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Resume from a previous checkpoint if this run was interrupted (e.g. Kaggle session timeout).
# NOTE: this only works within the SAME Kaggle session/output dir -- if you start a fresh
# committed run, /kaggle/working resets and there is nothing to resume from automatically.
resume_checkpoint = None
if os.path.exists(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[1]))
        resume_checkpoint = os.path.join(output_dir, checkpoints[-1])
        print(f"Resuming from: {resume_checkpoint}")
    else:
        print("No checkpoint found, starting fresh")

trainer.train(resume_from_checkpoint=resume_checkpoint)

## Check Training Progress

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
eval_rows = history[history["eval_f1"].notna()]
print(eval_rows[["epoch", "eval_loss", "eval_accuracy", "eval_f1"]])

## Save Final Model
**No test-set evaluation happens in this notebook.** Model + held-out test CSV go to `/kaggle/working/afroxlmr_sepedi_final` --
upload that whole folder as a Kaggle Model, then use it as Input in the separate `final_evaluation.ipynb` notebook.

In [ ]:
val_results = trainer.evaluate()
print("Validation results (NOT the final reportable test number):", val_results)

trainer.save_model("/kaggle/working/afroxlmr_sepedi_final")
tokenizer.save_pretrained("/kaggle/working/afroxlmr_sepedi_final")
print("Model + held_out_test_set.csv saved to /kaggle/working/afroxlmr_sepedi_final")
print("Next: upload this folder as a Kaggle Model, then run final_evaluation.ipynb")